In [1]:
try:
    # ignore ShapelyDeprecationWarning from fvcore
    from shapely.errors import ShapelyDeprecationWarning
    import warnings
    warnings.filterwarnings('ignore', category=ShapelyDeprecationWarning)
except:
    pass

import copy
import itertools
import logging
import os, json

from collections import OrderedDict
from typing import Any, Dict, List, Set

import torch

import detectron2.utils.comm as comm
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.config import get_cfg
from detectron2.data import MetadataCatalog
from detectron2.engine import (
    DefaultTrainer,
    default_argument_parser,
    default_setup,
    launch,
)
from detectron2.evaluation import (
    DatasetEvaluator,
    inference_on_dataset,
    print_csv_format,
    verify_results,
)
from detectron2.projects.deeplab import add_deeplab_config, build_lr_scheduler
from detectron2.solver.build import maybe_add_gradient_clipping
from detectron2.utils.logger import setup_logger

# MaskFormer
from mask2former import add_maskformer2_config
from mask2former_video import (
    YTVISDatasetMapper,
    YTVISEvaluator,
    add_maskformer2_video_config,
    build_detection_train_loader,
    build_detection_test_loader,
    get_detection_dataset_dicts,
)

/root/anaconda3/envs/syx_mask/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:


def setup(args):
    """
    Create configs and perform basic setups.
    """
    cfg = get_cfg()
    # for poly lr schedule
    add_deeplab_config(cfg)
    add_maskformer2_config(cfg)
    add_maskformer2_video_config(cfg)
    cfg.merge_from_file(args.config_file)
    cfg.merge_from_file("config.yaml")
    # cfg.merge_from_list(args.opts)
    cfg.MODEL.WEIGHTS = "./ckpt/train-30-2/model_00${iter}999.pth"
    cfg.freeze()
    default_setup(cfg, args)
    # Setup logger for "mask_former" module
    setup_logger(name="mask2former")
    setup_logger(output=cfg.OUTPUT_DIR, distributed_rank=comm.get_rank(), name="mask2former_video")
    # exit()
    return cfg


def main(args):
    cfg = setup(args)

    if args.eval_only:
        
        # test
        model = Trainer.build_model(cfg)
        DetectionCheckpointer(model, save_dir=cfg.OUTPUT_DIR).resume_or_load(
            cfg.MODEL.WEIGHTS, resume=args.resume
        )
        res = Trainer.test(cfg, model)
        if cfg.TEST.AUG.ENABLED:
            raise NotImplementedError
        if comm.is_main_process():
            verify_results(cfg, res)
        return res

    trainer = Trainer(cfg)
    trainer.resume_or_load(resume=args.resume)
    return trainer.train()


# if __name__ == "__main__":
#     args = default_argument_parser().parse_args()
#     print("Command Line Args:", args)
#     launch(
#         main,
#         args.num_gpus,
#         num_machines=args.num_machines,
#         machine_rank=args.machine_rank,
#         dist_url=args.dist_url,
#         args=(args,),
#     )


In [10]:
import argparse, sys
sys.argv = ['visualization.ipynb']
parser = argparse.ArgumentParser(description="Process some integers.")
parser.add_argument('--config-file', type=str, default="./configs/youtubevis_2019/swin/video_maskformer2_swin_tiny_bs16_8ep.yaml")
parser.add_argument('--num-gpus', type=int, default=2)
parser.add_argument('--eval-only', type=bool, default=True)
parser.add_argument('--scale', type=float, default=1.0, help='Scaling factor (default: 1.0)')
args = parser.parse_args()

cfg = setup(args)

Config 'config.yaml' has no VERSION. Assuming it to be compatible with latest v2.


[08/14 07:02:50 detectron2]: Rank of current process: 0. World size: 1
[08/14 07:02:51 detectron2]: Environment info:
-------------------------------  -----------------------------------------------------------------------------------
sys.platform                     linux
Python                           3.8.13 (default, Mar 28 2022, 11:38:47) [GCC 7.5.0]
numpy                            1.24.4
detectron2                       0.6 @/opt/data/private/syx/Mask2Former-envi/detectron2/detectron2
Compiler                         GCC 9.4
CUDA compiler                    CUDA 11.2
detectron2 arch flags            8.6
DETECTRON2_ENV_MODULE            <not set>
PyTorch                          1.9.0+cu111 @/root/anaconda3/envs/syx_mask/lib/python3.8/site-packages/torch
PyTorch debug build              False
torch._C._GLIBCXX_USE_CXX11_ABI  False
GPU available                    Yes
GPU 0,1,2,3                      NVIDIA GeForce RTX 4090 (arch=8.9)
Driver version                   525.105.17
C

In [12]:
model = build_model(cfg)

NameError: name 'build_model' is not defined